## Pipeline-Test auf den erstellten Testdaten (für YOLO)

In [3]:
!pip install jiwer

In [2]:
from jiwer import wer, cer

ref = "das ist ein test"
pred = ""

print("WER:", wer(ref, pred))
print("CER:", cer(ref, pred))

WER: 1.0
CER: 1.0


## Evaluation der Textzeilen (Gesamtpipeline)

In [4]:
import os
import re
import numpy as np
from jiwer import wer, cer

def read_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def normalize(text):
    return "\n".join(line.rstrip() for line in text.split("\n"))

def evaluate(gt, pred):
    return {
        "WER": wer(gt, pred),
        "CER": cer(gt, pred)
    }

pred_dir = "generierte_texte_maske/textzeilen/"
gt_dir = "ground_truth/textzeilen/"

gt_files = os.listdir(gt_dir)
pred_files = os.listdir(pred_dir)

def extract_id(filename):
    match = re.search(r"\d+", filename)
    return match.group() if match else None

gt_map = {extract_id(f): f for f in gt_files}
pred_map = {extract_id(f): f for f in pred_files}

common_ids = set(gt_map.keys()) & set(pred_map.keys())

print(f"Gefundene Paare: {len(common_ids)}")

results = []
all_gt_texts = []
all_pred_texts = []

for id_ in sorted(common_ids, key=int):
    gt_path = os.path.join(gt_dir, gt_map[id_])
    pred_path = os.path.join(pred_dir, pred_map[id_])

    gt_text = normalize(read_file(gt_path))
    pred_text = normalize(read_file(pred_path))

    res = evaluate(gt_text, pred_text)
    results.append(res)

    all_gt_texts.append(gt_text)
    all_pred_texts.append(pred_text)

    print(f"\n=== ID {id_} ===")
    print(res)

avg_wer = np.mean([r["WER"] for r in results])
avg_cer = np.mean([r["CER"] for r in results])

global_gt = "\n".join(all_gt_texts)
global_pred = "\n".join(all_pred_texts)

print("\n=== AVERAGE ===")
print("WER:", avg_wer)
print("CER:", avg_cer)

print("\n=== GLOBAL ===")
print("WER:", wer(global_gt, global_pred))
print("CER:", cer(global_gt, global_pred))

Gefundene Paare: 14

=== ID 0 ===
{'WER': 0.1724137931034483, 'CER': 0.036585365853658534}

=== ID 1 ===
{'WER': 0.2222222222222222, 'CER': 0.05963302752293578}

=== ID 2 ===
{'WER': 0.2857142857142857, 'CER': 0.05223880597014925}

=== ID 3 ===
{'WER': 0.375, 'CER': 0.04}

=== ID 4 ===
{'WER': 0.2413793103448276, 'CER': 0.045627376425855515}

=== ID 5 ===
{'WER': 0.5882352941176471, 'CER': 0.375}

=== ID 6 ===
{'WER': 0.4, 'CER': 0.21468926553672316}

=== ID 7 ===
{'WER': 0.4230769230769231, 'CER': 0.09540636042402827}

=== ID 8 ===
{'WER': 0.32142857142857145, 'CER': 0.13306451612903225}

=== ID 9 ===
{'WER': 0.14492753623188406, 'CER': 0.04569420035149385}

=== ID 10 ===
{'WER': 0.07692307692307693, 'CER': 0.00980392156862745}

=== ID 12 ===
{'WER': 0.5833333333333334, 'CER': 0.23333333333333334}

=== ID 13 ===
{'WER': 0.3076923076923077, 'CER': 0.06306306306306306}

=== ID 14 ===
{'WER': 0.0, 'CER': 0.0}

=== AVERAGE ===
WER: 0.29588190387060914
CER: 0.10029565972706432

=== GLOBAL 

In [5]:
from jiwer import wer, cer

def normalize(text):
    return "\n".join(line.rstrip() for line in text.split("\n"))

with open(r"E:\bachelor_projekte\box_mask_vergleich\ground_truth\textzeilen\pipe_test_mixed_12_text.txt", "r", encoding="utf-8") as f:
    gt = normalize(f.read())

with open(r"E:\bachelor_projekte\box_mask_vergleich\generierte_texte_maske\textzeilen\gen_text_12.txt", "r", encoding="utf-8") as f:
    pred = normalize(f.read())

print("WER:", wer(gt, pred))
print("CER:", cer(gt, pred))

WER: 0.5833333333333334
CER: 0.23333333333333334


## Evaluation der Formeln: Genauigkeit pro Zeile, sowie CER pro Zeile und global

In [3]:
import os
import re
import numpy as np
from jiwer import cer

def read_file(path):
    with open(path, "r", encoding="utf-8") as f:
        return f.read()

def normalize_whitespace(text):
    return "\n".join(
        " ".join(line.split()) for line in text.split("\n")
    )

def exact_match(gt, pred):
    return int(gt.strip() == pred.strip())

def extract_id(filename):
    match = re.search(r"\d+", filename)
    return match.group() if match else None

def levenshtein_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost  # substitution
            )

    return dp[m][n]


def exact_match_tol1(gt, pred):
    return int(levenshtein_distance(gt.strip(), pred.strip()) <= 1)

def exact_match_tol2(gt, pred):
    return int(levenshtein_distance(gt.strip(), pred.strip()) <= 2)


pred_dir = "generierte_texte_maske/mathe/"
gt_dir = "ground_truth/mathe_formeln/"

gt_files = os.listdir(gt_dir)
pred_files = os.listdir(pred_dir)

gt_map = {extract_id(f): f for f in gt_files}
pred_map = {extract_id(f): f for f in pred_files}

all_ids = set(gt_map.keys())

print(f"Gefundene Formel-Paare: {len(all_ids)}")

# Evaluation pro Zeile
line_results = []
all_gt_lines = []
all_pred_lines = []

total_empty_preds = 0

for id_ in sorted(all_ids, key=int):
    empty_in_file = 0
    gt_path = os.path.join(gt_dir, gt_map[id_])
    pred_path = os.path.join(pred_dir, pred_map[id_])

    gt_text = normalize_whitespace(read_file(gt_path))
    pred_text = normalize_whitespace(read_file(pred_path))

    gt_lines = gt_text.split("\n")
    pred_lines = pred_text.split("\n")

    max_len = max(len(gt_lines), len(pred_lines))

    print(f"\n=== ID {id_} ===")

    for i in range(max_len):
        gt_line = gt_lines[i] if i < len(gt_lines) else ""
        pred_line = pred_lines[i] if i < len(pred_lines) else ""

        # Whitespaces in den einzelnen Zeilen normalisieren
        gt_line = " ".join(gt_line.split())
        pred_line = " ".join(pred_line.split())

        # beide leer -> nicht evaluieren
        if gt_line.strip() == "" and pred_line.strip() == "":
            continue

        if gt_line.strip() != "" and pred_line.strip() != "":
            line_cer = cer(gt_line, pred_line)
        else:
            line_cer = 1.0

        if pred_line.strip() == "" and gt_line.strip() != "":  # leere Pred aber GT vorhanden
            empty_in_file += 1
            total_empty_preds += 1

        em = exact_match(gt_line, pred_line)
        em_tol1 = exact_match_tol1(gt_line, pred_line)
        em_tol2 = exact_match_tol2(gt_line, pred_line)

        line_results.append((id_, i, em, em_tol1, em_tol2))  # em_tol2 ergänzt

        all_gt_lines.append(gt_line)
        all_pred_lines.append(pred_line)

        print(f"Line {i}:")
        print(f"GT  : '{gt_line}'")
        print(f"PRED: '{pred_line}'")
        print(f"Exact Match: {em}")
        print(f"Exact Match (≤1 Fehler): {em_tol1}")
        print(f"Exact Match (≤2 Fehler): {em_tol2}")
        print(f"CER: {line_cer:.4f}")
        print()

# Metriken berechnen
avg_em = np.mean([r[2] for r in line_results]) if line_results else 0.0
avg_em_tol1 = np.mean([r[3] for r in line_results]) if line_results else 0.0
avg_em_tol2 = np.mean([r[4] for r in line_results]) if line_results else 0.0

global_gt = "\n".join(all_gt_lines)
global_pred = "\n".join(all_pred_lines)
global_cer = cer(global_gt, global_pred) if all_gt_lines else 0.0
print("\n--- ALL PRED LINES ---")

empty_pred_count = sum(1 for line in all_pred_lines if line == "")
print("Anzahl leerer Predictions in all_pred_lines:", empty_pred_count)

print("\n=== FORMULA RESULTS ===")
print("Exact Match Rate:", avg_em)
print("Exact Match (≤1 Fehler):", avg_em_tol1)
print("Exact Match (≤2 Fehler):", avg_em_tol2)
print("Global CER:", global_cer)
print("Anzahl Zeilen gesamt:", len(line_results))
print(f"\nLeere Predictions gesamt (GT vorhanden): {total_empty_preds}")

Gefundene Formel-Paare: 15

=== ID 0 ===
Line 0:
GT  : '$f(x)=x^{2}-2x+1$'
PRED: '$f(x)=x^{2}-2x+1$'
Exact Match: 1
Exact Match (≤1 Fehler): 1
Exact Match (≤2 Fehler): 1
CER: 0.0000

Line 1:
GT  : '$f(x)=(x-1)^{2}$'
PRED: '$f(x)=(x-1)^{2}$'
Exact Match: 1
Exact Match (≤1 Fehler): 1
Exact Match (≤2 Fehler): 1
CER: 0.0000

Line 2:
GT  : '$x=1$'
PRED: '$x=1$'
Exact Match: 1
Exact Match (≤1 Fehler): 1
Exact Match (≤2 Fehler): 1
CER: 0.0000

Line 3:
GT  : '$f(x)\ge0.$'
PRED: '$f(x)\ge0.$'
Exact Match: 1
Exact Match (≤1 Fehler): 1
Exact Match (≤2 Fehler): 1
CER: 0.0000


=== ID 1 ===
Line 0:
GT  : '$f(x)=x^{2}-4x+3$'
PRED: '$f(x)=x^{2}-4x+3$'
Exact Match: 1
Exact Match (≤1 Fehler): 1
Exact Match (≤2 Fehler): 1
CER: 0.0000

Line 1:
GT  : '$x=1$'
PRED: ''
Exact Match: 0
Exact Match (≤1 Fehler): 0
Exact Match (≤2 Fehler): 0
CER: 1.0000

Line 2:
GT  : '$f(1)=1^{2}-4\cdot1+3=0$'
PRED: '$f(1)=1^{2}-4\cdot1+3=0$'
Exact Match: 1
Exact Match (≤1 Fehler): 1
Exact Match (≤2 Fehler): 1
CER: 0.0000

Line

## Anzahl der Textzeilen und Formeln in den Ground Truth Dateien zählen

- Achtung: Da in den txt-Files manche Instanzen bzw. Erkennungen in einer Zeile stehen gibt es in, bspw. in den txt-Files 111 (Formula-Zeilen), obwohl es im Datensatz 128 Formulainstanzen gibt. Das gleiche gilt für die Textzeilen. Es könnte also sein, dass die Anzahl der Zeilen in den txt-Files nicht exakt mit der Anzahl der Instanzen im Datensatz übereinstimmt, da mehrere Instanzen in einer Zeile zusammengefasst sein können. (Also bspw. wenn mehrere Instanzen in einer Zeile erkannt wurden)

In [7]:
import os

textline_count = 0
formula_count = 0

folder = r"E:\own_dataset_htr\test_data_mixed\test_mixed_dataset\labels\train"  # Pfad zu deinen txt-Dateien anpassen

for filename in os.listdir(folder):
    if filename.endswith(".txt"):
        with open(os.path.join(folder, filename), "r") as f:
            for line in f:
                cls = line.strip().split()[0]
                if cls == "0":
                    textline_count += 1
                elif cls == "1":
                    formula_count += 1

print(f"Textline (0): {textline_count}")
print(f"Formula (1): {formula_count}")
print(f"Gesamt: {textline_count + formula_count}")

Textline (0): 106
Formula (1): 128
Gesamt: 234


In [8]:
import os

folder = r"E:\bachelor_projekte\box_mask_vergleich\ground_truth\textzeilen"

total_lines = 0

for filename in os.listdir(folder):
    if filename.endswith(".txt"):
        with open(os.path.join(folder, filename), "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f.readlines() if line.strip() != ""]
            total_lines += len(lines)

print(f"Gesamtanzahl Textzeilen: {total_lines}")

Gesamtanzahl Textzeilen: 91
